# 06 · RAG: Tanya Firaza (Side Quest)

**RAG (Retrieval-Augmented Generation)** = model bahasa menjawab dengan **membaca dokumen yang relevan lebih dulu**, bukan mengandalkan ingatannya. Ini mengurangi jawaban mengarang dan membuat jawaban bisa ditelusuri ke sumbernya.

```
pertanyaan → embedding → cari 3 dokumen terdekat (FAISS) → cukup relevan? → prompt + konteks → model bahasa → jawaban + sumber
                                                          └─ tidak → menolak menjawab, arahkan ke admin
```
Komponen: embedding **BAAI/bge-m3** (Hugging Face, via Ollama), vector database **FAISS**, generation **Qwen2.5-3B** lokal (via Ollama), cadangan Gemini.
**Prasyarat untuk menjalankan notebook ini:** Ollama berjalan, dan `ollama pull bge-m3`, `ollama pull qwen2.5:3b` sudah dilakukan.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

# Notebook ini berada di folder notebooks/; pindah ke root proyek agar path data/ dan src/ konsisten
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 170)
SEED = 42
print("Root proyek:", ROOT)

Root proyek: c:\Users\asus\firaza-custom-order


## 1. Basis pengetahuan
24 dokumen pendek berupa **contoh sintetis** (disusun dari brief Use Case 2 Firaza dan fitur aplikasi; bukan FAQ resmi). Hal yang tidak diketahui (harga, estimasi, kebijakan) ditulis "dikonfirmasi admin" dan tidak diisi angka karangan.

In [2]:
import json
kb = pd.DataFrame(json.load(open("data/knowledge_base.json", encoding="utf-8")))
print(len(kb), "dokumen")
kb[["id", "judul", "kategori", "isi"]].head(10)

24 dokumen


,id,judul,kategori,isi
0,kb01,Tentang Firaza,profil,Firaza adalah bisnis retail hampers dan personalized gifts. Produk utamanya handuk dan sajadah untuk kebut...
1,kb02,Kanal pembelian,profil,"Pembelian dilakukan terutama secara online lewat Shopee, TikTok Shop, dan website. Pesanan partai besar di..."
2,kb03,Produk utama,produk,"Kategori produk utama Firaza: hampers handuk, sajadah custom (printing atau bordir), kado pernikahan, dan ..."
3,kb04,Ready stock dan custom,produk,Ready stock adalah produk standar tanpa personalisasi yang bisa langsung dibeli. Custom adalah produk yang...
4,kb05,Sajadah custom,produk,Sajadah dapat dipersonalisasi dengan nama atau tulisan lewat printing atau bordir. Informasi yang perlu di...
5,kb06,Handuk custom,produk,"Handuk dapat dibordir dengan nama, inisial, atau logo. Informasi custom diteruskan ke vendor bordir, lalu ..."
6,kb07,Hampers,produk,"Hampers adalah paket kado berisi beberapa item, misalnya handuk dan sajadah, dengan kemasan dan bisa diser..."
7,kb08,Kartu ucapan,layanan,"Draf kartu ucapan dapat dibuat sesuai acara dan nada yang dipilih, misalnya hangat, formal, atau islami. N..."
8,kb09,Pesanan partai besar,layanan,"Untuk pesanan dalam jumlah banyak, misalnya souvenir pernikahan atau hampers kantor, pemesanan dilakukan l..."
9,kb10,Alur memesan produk custom,proses,"Alurnya: kirim pesan atau isi form di halaman pelanggan, lalu Anda mendapat kode pesanan. Admin mengonfirm..."


## 2. Embedding dan indeks FAISS
Tiap dokumen diubah menjadi vektor oleh bge-m3; pertanyaan juga. Vektor dinormalisasi sehingga **hasil kali dalam = cosine similarity**; FAISS (`IndexFlatIP`) mencari vektor terdekat. Indeks disimpan di `models/rag/` dan dibangun ulang otomatis bila isi basis pengetahuan berubah.

In [3]:
from rag_service import RagService, RagUnavailable
svc = RagService()
try:
    svc.ensure_index()
    print(svc.status())
except RagUnavailable as e:
    print("Ollama belum siap:", e)

{'dokumen': 24, 'embedding': 'ollama:bge-m3', 'indeks_siap': True, 'indeks_backend': 'numpy', 'llm_backend': 'ollama', 'llm_model': 'qwen2.5:3b', 'ambang_skor': 0.45}


## 3. Pencarian (retrieval)

In [4]:
pertanyaan = ["Gimana cara lihat pesanan saya sudah sampai mana?", "Berapa lama pengerjaan bordir?",
              "Boleh kirim logo perusahaan sendiri?", "Resep nasi goreng yang enak"]
rows = []
try:
    for q in pertanyaan:
        for rank, h in enumerate(svc.retrieve(q, k=3), 1):
            rows.append({"pertanyaan": q, "rank": rank, "dokumen": h["judul"], "skor": round(h["skor"], 3)})
    display(pd.DataFrame(rows))
except RagUnavailable as e:
    print("Ollama belum siap:", e)

,pertanyaan,rank,dokumen,skor
0,Gimana cara lihat pesanan saya sudah sampai mana?,1,Melacak pesanan,0.646
1,Gimana cara lihat pesanan saya sudah sampai mana?,2,Arti status pesanan,0.633
2,Gimana cara lihat pesanan saya sudah sampai mana?,3,Alur memesan produk custom,0.633
3,Berapa lama pengerjaan bordir?,1,Estimasi waktu pengerjaan,0.676
4,Berapa lama pengerjaan bordir?,2,Sajadah custom,0.527
5,Berapa lama pengerjaan bordir?,3,Handuk custom,0.523
6,Boleh kirim logo perusahaan sendiri?,1,Logo atau desain sendiri,0.673
7,Boleh kirim logo perusahaan sendiri?,2,Handuk custom,0.537
8,Boleh kirim logo perusahaan sendiri?,3,Menulis nama atau teks custom,0.537
9,Resep nasi goreng yang enak,1,Rekomendasi jenis pesanan oleh AI,0.359


Perhatikan dua hal: (1) dokumen yang benar muncul di peringkat atas walau kata-katanya tidak sama persis (embedding menangkap **makna**), dan (2) pertanyaan di luar domain ("resep nasi goreng") punya **skor teratas yang jauh lebih rendah**. Perbedaan skor inilah yang dipakai untuk menolak menjawab.

## 4. Menjawab (generation) dan menolak
Prompt memuat aturan (jawab hanya dari konteks, jangan mengarang harga dan waktu, sebut sumber), konteks bernomor, dan pertanyaan. Jika skor teratas di bawah ambang `RAG_MIN_SCORE`, sistem menolak.

In [5]:
for q in ["Bagaimana cara melacak pesanan saya?", "Berapa harga sajadah custom?", "Siapa presiden Indonesia sekarang?"]:
    try:
        out = svc.ask(q)
        print(f"Q: {q}\nmode: {out['mode']} | ditolak: {out['abstained']}\nA: {out['jawaban']}")
        print("sumber:", [(s['id'], s['skor']) for s in out['sumber']], "\n")
    except RagUnavailable as e:
        print("Ollama belum siap:", e); break

Q: Bagaimana cara melacak pesanan saya?
mode: ollama:qwen2.5:3b | ditolak: False
A: Gunakan menu Lacak Pesanan di halaman pelanggan. Masukkan kode pesanan dan nomor WhatsApp yang dipakai saat memesan untuk melihat tahap, item, dan riwayat pesanan. [1]
sumber: [('kb11', 0.735), ('kb10', 0.631), ('kb17', 0.61)] 

Q: Berapa harga sajadah custom?
mode: ollama:qwen2.5:3b | ditolak: False
A: Informasi harga sajadah custom tidak ada di konteks yang diberikan. Silakan hubungi admin untuk mendapatkan informasi tersebut. [1]
sumber: [('kb05', 0.616), ('kb22', 0.529), ('kb06', 0.484)] 

Q: Siapa presiden Indonesia sekarang?
mode: abstain | ditolak: True
A: Maaf, saya belum menemukan informasi tentang itu di panduan Firaza. Silakan hubungi admin Firaza lewat WhatsApp atau Instagram resmi untuk bantuan lebih lanjut.
sumber: [] 



Pertanyaan harga dijawab dengan "dikonfirmasi admin" (sesuai isi dokumen) dan **tidak** dengan angka karangan; pertanyaan di luar domain ditolak.

## 5. Evaluasi terukur
28 pertanyaan: 22 dalam domain (dengan dokumen jawaban yang diharapkan) dan 6 di luar domain. **hit@k** = proporsi pertanyaan yang dokumen benarnya ada di k hasil teratas; **MRR** = rata-rata 1/peringkat dokumen benar.

In [6]:
from eval_rag import evaluate
try:
    res = evaluate(svc)
    show = {k: res[k] for k in ["embedding", "indeks", "hit@1", "hit@3", "mrr", "ambang_saat_ini",
                                 "ditolak_benar_luar_domain", "ditolak_salah_dalam_domain", "ambang_disarankan"]}
    display(pd.Series(show, name="hasil"))
    print("skor teratas dalam domain :", res["skor_teratas_dalam_domain"])
    print("skor teratas luar domain  :", res["skor_teratas_luar_domain"])
    print("pertanyaan yang gagal hit@3:", res["gagal_hit@3"])
except RagUnavailable as e:
    print("Ollama belum siap:", e)

embedding                     ollama:bge-m3
indeks                                numpy
hit@1                                 0.909
hit@3                                 0.955
mrr                                   0.932
ambang_saat_ini                        0.45
ditolak_benar_luar_domain               1.0
ditolak_salah_dalam_domain              0.0
ambang_disarankan                     0.469
Name: hasil, dtype: object

skor teratas dalam domain : {'min': 0.53, 'rata2': 0.664}
skor teratas luar domain  : {'maks': 0.408, 'rata2': 0.369}
pertanyaan yang gagal hit@3: [{'pertanyaan': 'Berapa harga sajadah custom?', 'diharapkan': ['kb15'], 'teratas': 'kb05'}]


## 6. Mengapa begini (untuk presentasi)

- **Mengapa RAG, bukan langsung bertanya ke model?** Model kecil mudah mengarang soal harga dan kebijakan toko; RAG membatasinya pada dokumen yang kita sediakan dan menampilkan sumbernya.
- **Mengapa bge-m3?** Embedding multibahasa (termasuk Indonesia) dari Hugging Face yang berjalan lokal.
- **Mengapa FAISS?** Cepat, ringan, tanpa server; untuk 24 dokumen indeks sederhana sudah cukup.
- **Mengapa model lokal Qwen2.5-3B?** Muat penuh di GPU 4 GB (kuantisasi 4-bit), data tidak keluar dari komputer, dan brief memberi nilai tambah untuk model lokal. Gemini jadi cadangan.
- **Mengapa menolak menjawab?** Lebih baik jujur "tidak tahu" daripada jawaban salah dengan nada yakin.

**Keterbatasan:** basis pengetahuan contoh dan kecil; ambang skor dikalibrasi pada 28 pertanyaan; model 3B masih bisa keliru sehingga hal penting perlu dicek manusia.